In [2]:
# https://chatgpt.com/c/68120d89-7490-8002-a4e7-9fb42841216e
# https://chatgpt.com/c/682384e9-7f3c-8002-9ad7-f828c84e8759

# Variantas:  failas su triukšmų fragmentų laikais "startTime, "endTime" 

# Here’s the full, modular script combining everything for running inside a Jupyter Notebook. It:

# Parses noise intervals from List_of_noises_in_zive_records.txt
# Backs up .json files if they need updates
# Updates "noises" → [] and inserts "noises_annotated" only if not already present with identical content
# Updates the "noni" column in the Excel file
# Saves updated Excel as visi_zive_irasai_new2.xlsx

# You will get:

# ✅ Updated .json files (only when needed)
# ✅ Backup of originals in backup_jsons2/
# ✅ Corrected noni values in Excel
# ✅ Updated Excel saved as visi_zive_irasai.xlsx

# Ima duomenis testavimui iš DUOMENYS_TST/records_npy_all failų List_of_noises_in_zive_records.txt, visi_zive_irasai.xlsx
# ir apdoroja testavimui įrašus: 
# fileName = '1004_0.npy', '1005_4.npy', '1006_1.npy', '1006_2.npy', '1006_3.npy' - egzistuoja
# fileName = '1006_4.npy' - neegzistuoja


import json
import re
import shutil
import pandas as pd
from pathlib import Path
import numpy as np

# === CONSTANTS ===
fs = 200  # sampling frequency

# === UPDATED LOADER FOR TIME-BASED INTERVALS ===
# def load_noise_intervals_from_secs(text_path, fs=200):
#     with text_path.open("r", encoding="utf-8") as f:
#         lines = f.readlines()
#     cleaned_lines = [line for line in lines if not line.strip().startswith("#") and line.strip()]
#     content = "".join(cleaned_lines)

#     pattern = r"fileName\s*=\s*'([^']+)'\s*\[\s*(.*?)\s*\]"
#     matches = re.findall(pattern, content, re.DOTALL)

#     noise_map = {}
#     for file_name, time_block in matches:
#         base = file_name.split('.')[0]
#         try:
#             time_list = json.loads(f"[{time_block}]")
#             noise_list = [
#                 {
#                     "startIndex": int(round(d["startTime"] * fs)),
#                     "endIndex": int(round(d["endTime"] * fs))
#                 }
#                 for d in time_list
#             ]
#             noise_map[base] = noise_list
#         except json.JSONDecodeError as e:
#             print(f"❌ Error parsing time block in {file_name}: {e}")
#     return noise_map


def load_noise_intervals_from_secs(text_path, fs=200, data_dir=None):
    """
    Load and convert noise intervals from seconds to sample indices.
    Ensures that endIndex < ecg_length from actual .npy file.
    """
    if data_dir is None:
        raise ValueError("data_dir must be provided to locate .npy files.")

    with text_path.open("r", encoding="utf-8") as f:
        lines = f.readlines()
    cleaned_lines = [line for line in lines if not line.strip().startswith("#") and line.strip()]
    content = "".join(cleaned_lines)

    pattern = r"fileName\s*=\s*'([^']+)'\s*\[\s*(.*?)\s*\]"
    matches = re.findall(pattern, content, re.DOTALL)

    noise_map = {}

    for file_name, time_block in matches:
        base = file_name.split('.')[0]
        npy_path = data_dir / file_name

        if not npy_path.exists():
            print(f"⚠️ ECG file not found: {file_name}. Skipping.")
            continue

        try:
            ecg_signal = np.load(npy_path)
            ecg_length = len(ecg_signal)
        except Exception as e:
            print(f"❌ Error reading {file_name}: {e}")
            continue

        try:
            time_list = json.loads(f"[{time_block}]")
            noise_list = []
            for d in time_list:
                start_idx = int(round(d["startTime"] * fs))
                end_idx = int(round(d["endTime"] * fs))
                if end_idx >= ecg_length:
                    print(f"⚠️ Adjusting endIndex {end_idx} to {ecg_length - 1} for {file_name}")
                    end_idx = ecg_length - 1
                noise_list.append({"startIndex": start_idx, "endIndex": end_idx})
            noise_map[base] = noise_list
        except json.JSONDecodeError as e:
            print(f"❌ Error parsing time block in {file_name}: {e}")
            continue

    return noise_map


def update_json_and_excel(noise_data, df, json_dir, backup_dir):
    modified_files = []

    for base, new_noises in noise_data.items():
        print(f"\n🔍 Processing {base}...")
        print(f"🔍 New noises annotated: {new_noises}")

        json_path = json_dir / f"{base}.json"
        if not json_path.exists():
            print(f"⚠️  {json_path.name} not found. Skipping.")
            continue

        try:
            with json_path.open("r", encoding="utf-8") as f:
                data = json.load(f)
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error in {json_path.name}: {e}")
            continue

        existing_annotated = data.get("noises_annotated", None)

        raw_noni = df.loc[df["filename"] == base, "noni"].values[0]
        if str(raw_noni).strip().upper() == "NA":
            current_noni = None
        else:
            current_noni = int(raw_noni)

        print(f"🔍 Existing noises annotated: {existing_annotated}")
        print(f"🔍 Current 'noni' in Excel: {current_noni}")

        if existing_annotated == new_noises and current_noni == len(new_noises):
            print(f"🟡 {json_path.name}: Already up-to-date. Skipping.")
            continue

        shutil.copy(json_path, backup_dir / f"{base}.json")
        print(f"🔄 Backed up {json_path.name} to {backup_dir.name}/")

        data["noises"] = []
        data["noises_annotated"] = new_noises
        with json_path.open("w", encoding="utf-8") as f:
            json.dump(data, f, indent=4)
        print(f"✅ Updated {json_path.name}")

        if base in df["filename"].values:
            df.loc[df["filename"] == base, "noni"] = len(new_noises)
            print(f"📝 Updated 'noni' in Excel for {base} to {len(new_noises)}")
        else:
            print(f"⚠️  Excel row for {base} not found.")

        modified_files.append(base)

    return modified_files

# start0
# === PATH SETUP ===
project_root = Path.cwd().parent  # Adjust if needed
data_dir = project_root / "DUOMENYS_UPD" / "records_npy_all"
list_file = data_dir / "List_of_noises_secs_in_zive_records.txt" # failas su triukšmų fragmentų laikais "startTime, "endTime" 
json_backup_dir = data_dir / "backup_jsons2"
excel_file = data_dir / "visi_zive_irasai.xlsx"
excel_output = excel_file

json_backup_dir.mkdir(exist_ok=True)

# === MAIN WORKFLOW ===
noise_data = load_noise_intervals_from_secs(list_file, fs=fs, data_dir=data_dir)
print(f"✅ Loaded noise data for {len(noise_data)} records.")

list_of_files = [f"{base}" for base in noise_data.keys()]
print("\nlist of processing files:", list_of_files)

df_excel = pd.read_excel(excel_file, keep_default_na=False)
print(f"✅ Excel loaded with {len(df_excel)} rows.")

updated = update_json_and_excel(noise_data, df_excel, data_dir, json_backup_dir)
print(f"\n📦 Modified {len(updated)} JSON files.")

df_excel.to_excel(excel_output, index=False)
print(f"📄 Excel saved to: {excel_output}")



⚠️ Adjusting endIndex 128000 to 127998 for 1001_2.npy
✅ Loaded noise data for 42 records.

list of processing files: ['1001_2', '1001_3', '1001_5', '1001_6', '1001_7', '1001_8', '1002_1', '1004_0', '1005_0', '1005_4', '1006_1', '1006_2', '1006_3', '1007_1', '1008_0', '1008_12', '1009_12', '1009_13', '1010_0', '1011_0', '1011_1', '1013_13', '1013_14', '1015_0', '1015_17', '1017_0', '1018_6', '1018_7', '1019_0', '1019_1', '1023_0', '1024_0', '1025_0', '1026_0', '1026_4', '1027_0', '1028_1', '1029_0', '1030_5', '1097_0', '1097_1', '1102_0']
✅ Excel loaded with 1092 rows.

🔍 Processing 1001_2...
🔍 New noises annotated: [{'startIndex': 0, 'endIndex': 300}, {'startIndex': 1000, 'endIndex': 1200}, {'startIndex': 3900, 'endIndex': 5100}, {'startIndex': 5800, 'endIndex': 6400}, {'startIndex': 9200, 'endIndex': 10600}, {'startIndex': 11040, 'endIndex': 11280}, {'startIndex': 13400, 'endIndex': 13800}, {'startIndex': 15720, 'endIndex': 16400}, {'startIndex': 24400, 'endIndex': 33900}, {'startInde